In [1]:
import numpy as np, pandas as pd 
from sklearn.preprocessing import StandardScaler

def read_csv(file, names=[]):
    index_names = ['unit_number', 'time_cycles']
    setting_names = ['setting_1', 'setting_2', 'setting_3']
    sensor_names = ['s_{}'.format(i+1) for i in range(0,21)]
    col_names = index_names + setting_names + sensor_names
    df = pd.read_csv(f'CMAPSSData/{file}' ,sep='\s+',header=None,index_col=False,names=col_names if names == [] else names)
    return df

train1 = read_csv("train_FD001.txt")
test1 = read_csv("test_FD001.txt")
rul1 = read_csv("RUL_FD001.txt", ['RUL'])

<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2096580/3002517647.py:9: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(f'CMAPSSData/{file}' ,sep='\s+',header=None,index_col=False,names=col_names if names == [] else names)


In [2]:
train1.isnull().sum()

unit_number    0
time_cycles    0
setting_1      0
setting_2      0
setting_3      0
s_1            0
s_2            0
s_3            0
s_4            0
s_5            0
s_6            0
s_7            0
s_8            0
s_9            0
s_10           0
s_11           0
s_12           0
s_13           0
s_14           0
s_15           0
s_16           0
s_17           0
s_18           0
s_19           0
s_20           0
s_21           0
dtype: int64

In [3]:
summary = train1.groupby('unit_number')['time_cycles'].agg(
    mean_cycle  = 'mean',
    last_cycle   = 'max',
    total_cycles = 'count'   # same as max since cycles start at 1
).reset_index()

In [4]:
summary

,unit_number,mean_cycle,last_cycle,total_cycles
0,1,96.5,192,192
1,2,144.0,287,287
2,3,90.0,179,179
3,4,95.0,189,189
4,5,135.0,269,269
...,...,...,...,...
95,96,168.5,336,336
96,97,101.5,202,202
97,98,78.5,156,156
98,99,93.0,185,185


# Statistical analysis to find anomaly

check how many are negative values.

In [5]:
((train1.groupby("unit_number")["time_cycles"].max() - (test1.groupby("unit_number")["time_cycles"].max() + rul1.RUL.max())) > 0).value_counts()

time_cycles
False    82
True     18
Name: count, dtype: int64

82 negative values. Invalid for anomaly threshold.

check if mean of (test max + rul) larger than the train max

In [6]:
((test1.groupby("unit_number")["time_cycles"].max() + rul1.RUL.max()).mean() > train1.groupby("unit_number")["time_cycles"].max()).value_counts()

time_cycles
True     90
False    10
Name: count, dtype: int64

90 of them are larger than train max which is invalid.

Same thing but in median value

In [7]:
((test1.groupby("unit_number")["time_cycles"].max() + rul1.RUL.max()).median() > train1.groupby("unit_number")["time_cycles"].max()).value_counts()

time_cycles
True     92
False     8
Name: count, dtype: int64

92 are larger, invalid

# Claude Suggestions:

In the end Claude explained using the total run cycles - current cycle > 30 to be anomaly

In [8]:
def output_csv_w_label(df, name = ""):
    # compute RUL per row
    max_cycles = df.groupby('unit_number')['time_cycles'].max().reset_index()
    max_cycles.columns = ['unit_number', 'max_cycle']
    new_df = df.merge(max_cycles, on='unit_number')

    new_df['RUL']   = new_df['max_cycle'] - new_df['time_cycles']
    new_df['label'] = (new_df['RUL'] <= 30).astype(int)   # 0 = normal, 1 = anomaly
    new_df.to_csv(f"CMAPSSData/{name}.csv", index=False)

In [10]:
train1 = read_csv("train_FD001.txt")
train2 = read_csv("train_FD002.txt")
train3 = read_csv("train_FD003.txt")
train4 = read_csv("train_FD004.txt")

test1 = read_csv("test_FD001.txt")
test2 = read_csv("test_FD002.txt")
test3 = read_csv("test_FD003.txt")
test4 = read_csv("test_FD004.txt")

output_csv_w_label(train1, name="train_FD001_w_label")
output_csv_w_label(train2, name="train_FD002_w_label")
output_csv_w_label(train3, name="train_FD003_w_label")
output_csv_w_label(train4, name="train_FD004_w_label")
output_csv_w_label(test1, name="test_FD001_w_label")
output_csv_w_label(test2, name="test_FD002_w_label")
output_csv_w_label(test3, name="test_FD003_w_label")
output_csv_w_label(test4, name="test_FD004_w_label")
